# 07 — Sentinel-1 SLC Data Acquisition from CDSE

**Author:** Florian Klaver

**Prerequisite:** Notebook 02 must have run and `data/temporal_matches.csv` must exist.

---

## Why SLC? Why coherence?

Notebook 03 computes **backscatter** features from GRD products. The S1-only model
achieves F1 ≈ 0.5 (barely above random) — GRD backscatter alone is a weak mowing signal.

**InSAR coherence** is fundamentally different: it measures the *phase correlation* between
two SAR acquisitions from the same orbit. Over vegetation:
- **Before mowing**: tall, moving grass causes strong temporal decorrelation → low coherence (~0.2–0.4)
- **After mowing**: short, stable stubble shows much higher coherence (~0.5–0.8)
- **coherence_jump = coherence_after − coherence_before** is the primary mowing signal

This notebook acquires the SLC scenes needed to compute coherence for all 92 events.

## Temporal logic: SLC coherence triplet

For each mowing event we need **three SLC acquisitions** from the **same relative orbit**:

```
SLC_t1 (before_far_slc)  ──────────── 12-day baseline ──────────── SLC_t2 (before_near_slc)
                                                                         │
                                                                    12-day baseline
                                                                         │
                                                                    SLC_t3 (after_slc)

coherence_before = coh(SLC_t1, SLC_t2)   → low if grass is growing
coherence_after  = coh(SLC_t2, SLC_t3)   → high if grass was recently cut
coherence_jump   = coherence_after − coherence_before   ← key feature
```

**Critical constraint:** both scenes in a coherence pair must be from the **same relative orbit**
and same subswath. Mixing orbits destroys phase coherence.

## Disk space management

One S1 IW SLC product ≈ 4 GB. With ~110 unique scenes we need ~440 GB, which exceeds
available disk space. **Strategy: batch download → process → delete raw SLCs → repeat.**

This notebook handles only steps 1–5 (query, plan, and batch download).
Notebook 08 does the SNAP processing. After processing a batch, manually delete raw SLCs
from `data/Sentinel_S1_SLC/` before downloading the next batch.

## Notebook workflow

1. Define AOI (from S2 reference tile, same as nb02)
2. Configure CDSE credentials
3. Query the full S1 IW SLC catalogue over the AOI (metadata only, no download)
4. For each event, find the best same-orbit SLC triplet
5. Build `data/slc_scene_index.csv` and `data/slc_event_coverage.csv`
6. Batch download of SLC scenes (with skip logic for already-downloaded files)

---
## 1. Setup

In [21]:
import os
import glob
import json
import configparser
import numpy as np
import pandas as pd
import rasterio
from pyproj import Transformer
from datetime import timedelta

from cdsetool.credentials import Credentials
from cdsetool.query import query_features, describe_collection
from cdsetool.download import download_features
from dotenv import load_dotenv
load_dotenv()

# ── Paths ──────────────────────────────────────────────────────────────────
S2_DIR            = r'..\data\Sentinel_CH'
TEMPORAL_MATCHES  = r'..\data\temporal_matches.csv'
SLC_DIR           = r'..\data\Sentinel_S1_SLC'
SLC_INDEX_PATH    = r'..\data\slc_scene_index.csv'
SLC_COVERAGE_PATH = r'..\data\slc_event_coverage.csv'
COH_DIR           = r'..\data\features_coherence'

os.makedirs(SLC_DIR, exist_ok=True)

# ── SNAP configuration (one-time setup for pyroSAR) ────────────────────────
SNAP_HOME = r'C:\Program Files\esa-snap'
_pyrosar_cfg = os.path.join(os.path.expanduser('~'), '.pyrosar', 'config.ini')
if not os.path.exists(_pyrosar_cfg):
    os.makedirs(os.path.dirname(_pyrosar_cfg), exist_ok=True)
    _cfg = configparser.RawConfigParser()
    _cfg.add_section('SNAP')
    _cfg.set('SNAP', 'path', os.path.join(SNAP_HOME, 'bin', 'snap64.exe'))
    _cfg.set('SNAP', 'gpt',  os.path.join(SNAP_HOME, 'bin', 'gpt.exe'))
    _cfg.set('SNAP', 'etc',  os.path.join(SNAP_HOME, 'etc'))
    with open(_pyrosar_cfg, 'w') as f:
        _cfg.write(f)
    print(f'pyroSAR config written: {_pyrosar_cfg}')
else:
    print(f'pyroSAR config exists:  {_pyrosar_cfg}')

# ── Parameters ─────────────────────────────────────────────────────────────
# S1 repeat cycle at Zürich: 12 days (Sentinel-1A only post-2022; 6 days with A+B)
# We accept 6 or 12 day baselines.
VALID_BASELINES_DAYS = [6, 12]
# Search window around each S2 date when looking for a matching SLC
# SLC triplet dates won't coincide exactly with S2 dates — we search within ±8 days
SLC_MATCH_TOLERANCE = 8

print('Setup complete.')

pyroSAR config exists:  C:\Users\flori\.pyrosar\config.ini
Setup complete.


---
## 2. CDSE Credentials

Enter your Copernicus Data Space Ecosystem username and password.
These are the same credentials used for the Copernicus Browser.

For security, set them as environment variables before running:
```bash
set CDSE_USERNAME=your_email@example.com
set CDSE_PASSWORD=your_password
```
Or enter them directly in the cell below (do not commit to git).

In [22]:
CDSE_USERNAME = os.environ.get('CDSE_USERNAME', '')  # or hardcode: 'your@email.com'
CDSE_PASSWORD = os.environ.get('CDSE_PASSWORD', '')  # or hardcode: 'your_password'

if not CDSE_USERNAME or not CDSE_PASSWORD:
    raise ValueError(
        'CDSE credentials not set. '
        'Set environment variables CDSE_USERNAME and CDSE_PASSWORD, '
        'or hardcode them in this cell.'
    )

credentials = Credentials(username=CDSE_USERNAME, password=CDSE_PASSWORD)
# Verify credentials by opening a session
session = credentials.get_session()
print(f'CDSE credentials verified. Session: {type(session).__name__}')

CDSE credentials verified. Session: Session


---
## 3. Define Area of Interest

Same AOI derivation as notebook 02 — from the S2 reference tile bounding box.

In [23]:
s2_files = sorted(glob.glob(os.path.join(S2_DIR, '*.tif')))
if not s2_files:
    raise FileNotFoundError(f'No S2 GeoTIFFs found in {S2_DIR}')

with rasterio.open(s2_files[0]) as s2:
    b = s2.bounds  # EPSG:2056

transformer = Transformer.from_crs('EPSG:2056', 'EPSG:4326', always_xy=True)
lon_min, lat_min = transformer.transform(b.left,  b.bottom)
lon_max, lat_max = transformer.transform(b.right, b.top)

# WKT polygon for CDSE query
aoi_wkt = (
    f'POLYGON(('
    f'{lon_min:.5f} {lat_min:.5f}, '
    f'{lon_max:.5f} {lat_min:.5f}, '
    f'{lon_max:.5f} {lat_max:.5f}, '
    f'{lon_min:.5f} {lat_max:.5f}, '
    f'{lon_min:.5f} {lat_min:.5f}'
    f'))'
)

print(f'S2 reference tile: {os.path.basename(s2_files[0])}')
print(f'AOI (EPSG:2056):   X=[{b.left:.1f}, {b.right:.1f}]  Y=[{b.bottom:.1f}, {b.top:.1f}]')
print(f'AOI (WGS84):       lon=[{lon_min:.5f}, {lon_max:.5f}]  lat=[{lat_min:.5f}, {lat_max:.5f}]')
print(f'AOI WKT:           {aoi_wkt}')

S2 reference tile: 2019-03-01.tif
AOI (EPSG:2056):   X=[2682012.9, 2685824.6]  Y=[1254788.2, 1260380.7]
AOI (WGS84):       lon=[8.52587, 8.57747]  lat=[47.43878, 47.48859]
AOI WKT:           POLYGON((8.52587 47.43878, 8.57747 47.43878, 8.57747 47.48859, 8.52587 47.48859, 8.52587 47.43878))


---
## 4. Query CDSE Catalogue (metadata only)

Fetch all S1 IW SLC ascending products over the study area from 2019–2023.
This is a metadata-only query — no data is downloaded yet.

We query the entire date range once and cache the results locally to avoid
repeated API calls during the pairing logic.

In [24]:
CATALOGUE_CACHE = r'..\data\slc_catalogue_cache.json'

if os.path.exists(CATALOGUE_CACHE):
    print('Loading catalogue from cache...')
    with open(CATALOGUE_CACHE, 'r') as f:
        catalogue_raw = json.load(f)
else:
    print('Querying CDSE catalogue (ascending IW SLC, 2019-01-01 to 2023-12-31)...')
    print('This may take 1-2 minutes.')
    # query_features uses the public OData API — no credentials needed for metadata queries
    features = list(query_features(
        'SENTINEL-1',
        {
            'contentDateStartGe': '2019-01-01',
            'contentDateEndLe':   '2023-12-31',
            'productType':        'IW_SLC__1S',
            'operationalMode':    'IW',
            'orbitDirection':     'ASCENDING',
            'geometry':           aoi_wkt,
        },
    ))
    catalogue_raw = features
    with open(CATALOGUE_CACHE, 'w') as f:
        json.dump(catalogue_raw, f)
    print(f'Saved {len(catalogue_raw)} results to cache.')

print(f'\nTotal S1 IW SLC ascending products over AOI (2019-2023): {len(catalogue_raw)}')
if catalogue_raw:
    sample = catalogue_raw[0]
    # OData API returns flat dict with capitalized keys (Id, Name, ContentDate, ...)
    print(f'Sample product ID: {sample["Id"]}')
    print(f'Sample name:       {sample["Name"]}')

Loading catalogue from cache...

Total S1 IW SLC ascending products over AOI (2019-2023): 633
Sample product ID: b014c03b-cdef-5ce9-85b3-ce6f72585f23
Sample name:       S1A_IW_SLC__1SDV_20190101T171515_20190101T171542_025287_02CC09_0A0B.SAFE


In [25]:
# Parse catalogue into a structured DataFrame
# OData response format: flat dict with keys Id, Name, ContentDate, GeoFootprint, ...
# Name format: S1A_IW_SLC__1SDV_20190101T171515_20190101T171542_025287_02CC09_XXXX.SAFE
# Split by '_': [0]=S1A [1]=IW [2]=SLC [3]='' [4]=1SDV [5]=date1 [6]=date2 [7]=abs_orbit
#                                       ^ empty string from double-underscore SLC__1SDV

def _rel_orbit(abs_orbit: int, platform: str) -> int:
    """Convert S1 absolute orbit to relative orbit number."""
    if 'S1A' in platform:
        return (abs_orbit - 73) % 175 + 1
    elif 'S1B' in platform:
        return (abs_orbit - 27) % 175 + 1
    return -1


records = []
for feat in catalogue_raw:
    product_id = feat['Id']
    name       = feat['Name'].replace('.SAFE', '')  # strip extension

    # Date from ContentDate.Start (ISO-8601)
    start_str  = feat.get('ContentDate', {}).get('Start', '')

    # Parse name components
    parts    = name.split('_')
    platform = parts[0] if len(parts) > 0 else ''       # S1A / S1B
    try:
        abs_orbit = int(parts[7])                        # index 7 due to double-underscore in SLC__1SDV
    except (IndexError, ValueError):
        abs_orbit = -1

    orbit_rel = _rel_orbit(abs_orbit, platform) if abs_orbit > 0 else -1

    records.append({
        'product_id': product_id,
        'name':       name + '.SAFE',                    # keep .SAFE suffix for file matching
        'date':       pd.to_datetime(start_str[:10]) if start_str else pd.NaT,
        'orbit_rel':  orbit_rel,
        'orbit_abs':  abs_orbit,
        'platform':   platform,
    })

slc_catalogue = (
    pd.DataFrame(records)
    .dropna(subset=['date'])
    .sort_values('date')
    .reset_index(drop=True)
)
slc_catalogue['date'] = pd.to_datetime(slc_catalogue['date']).dt.normalize()

print(f'Parsed catalogue: {len(slc_catalogue)} scenes')
print(f'Date range: {slc_catalogue["date"].min().date()} to {slc_catalogue["date"].max().date()}')
print(f'Unique relative orbits: {sorted(slc_catalogue["orbit_rel"].unique())}')
print(f'Unique platforms: {sorted(slc_catalogue["platform"].unique())}')
print('\nSample rows:')
print(slc_catalogue.head(8).to_string(index=False))

Parsed catalogue: 633 scenes
Date range: 2019-01-01 to 2023-12-30
Unique relative orbits: [np.int64(15), np.int64(88)]
Unique platforms: ['S1A', 'S1B']

Sample rows:
                          product_id                                                                     name       date  orbit_rel  orbit_abs platform
b014c03b-cdef-5ce9-85b3-ce6f72585f23 S1A_IW_SLC__1SDV_20190101T171515_20190101T171542_025287_02CC09_0A0B.SAFE 2019-01-01         15      25287      S1A
13a35855-34d7-52e4-8630-9d5e9e5e244c S1A_IW_SLC__1SDV_20190101T171539_20190101T171606_025287_02CC09_C6BF.SAFE 2019-01-01         15      25287      S1A
0f917fb3-9f1a-5576-8ddb-ed0b817731a3 S1A_IW_SLC__1SDV_20190106T172345_20190106T172412_025360_02CEAE_E0E9.SAFE 2019-01-06         88      25360      S1A
c9f52f65-38bc-58cb-aa94-294ee5b79cf2 S1B_IW_SLC__1SDV_20190107T171445_20190107T171512_014391_01AC8B_AFEA.SAFE 2019-01-07         15      14391      S1B
aeeab18a-bf11-5a00-8bb1-e08c46002713 S1B_IW_SLC__1SDV_20190112T172256_2019

---
## 5. Determine Target Relative Orbit

For coherence to work, all scenes in a pair must share the **same relative orbit number**.
We inspect which relative orbit(s) cover the study area and pick the one with the best
temporal coverage over 2019–2023.

At Zürich Airport (lon≈8.55°, lat≈47.46°), typically one or two ascending orbit tracks
cover the area.

In [26]:
orbit_stats = (
    slc_catalogue
    .groupby('orbit_rel')
    .agg(
        n_scenes=('date', 'count'),
        first_date=('date', 'min'),
        last_date=('date', 'max'),
    )
    .sort_values('n_scenes', ascending=False)
)

print('=== Ascending orbit tracks over study area ===')
print(orbit_stats.to_string())

# Pick the orbit with the most scenes (best temporal sampling)
TARGET_ORBIT = int(orbit_stats.index[0])
print(f'\nSelected relative orbit: {TARGET_ORBIT}  ({orbit_stats.loc[TARGET_ORBIT, "n_scenes"]} scenes)')

# Filter catalogue to selected orbit
orbit_catalogue = slc_catalogue[slc_catalogue['orbit_rel'] == TARGET_ORBIT].copy().reset_index(drop=True)

# Compute temporal baselines between consecutive acquisitions
orbit_catalogue['baseline_to_prev'] = orbit_catalogue['date'].diff().dt.days

print(f'\nBaseline distribution for orbit {TARGET_ORBIT}:')
print(orbit_catalogue['baseline_to_prev'].dropna().value_counts().sort_index())

=== Ascending orbit tracks over study area ===
           n_scenes first_date  last_date
orbit_rel                                
15              387 2019-01-01 2023-12-30
88              246 2019-01-06 2023-12-23

Selected relative orbit: 15  (387 scenes)

Baseline distribution for orbit 15:
baseline_to_prev
0.0     154
6.0     160
12.0     72
Name: count, dtype: int64


---
## 6. Match SLC Triplets to Mowing Events

For each mowing event, find the three SLC acquisitions (from the target orbit) that:
1. Bracket the S2 `before_near` date → gives SLC_t2
2. Are adjacent to SLC_t2 with a 6 or 12 day baseline → gives SLC_t1 (before) and SLC_t3 (after)

Matching strategy:
- Find all SLC acquisitions within `±SLC_MATCH_TOLERANCE` days of each S2 triplet date
- Prefer the acquisition closest to the S2 date
- Verify that consecutive SLC pairs have a valid temporal baseline (6 or 12 days)

In [27]:
matches_df = pd.read_csv(TEMPORAL_MATCHES)
matches_df['event_date']       = pd.to_datetime(matches_df['event_date'])
matches_df['before_far_date']  = pd.to_datetime(matches_df['before_far_file'].str.replace('.tif', '', regex=False))
matches_df['before_near_date'] = pd.to_datetime(matches_df['before_near_file'].str.replace('.tif', '', regex=False))
matches_df['after_date']       = pd.to_datetime(matches_df['after_file'].str.replace('.tif', '', regex=False))
matches_df['match_id']         = matches_df.index

orbit_dates = orbit_catalogue['date'].values  # numpy array of datetime64
orbit_index  = orbit_catalogue.set_index('date')

def find_nearest_slc(target_date, tolerance_days=SLC_MATCH_TOLERANCE):
    """Return the orbit_catalogue row whose date is closest to target_date within tolerance."""
    target = pd.Timestamp(target_date)
    window = orbit_catalogue[
        (orbit_catalogue['date'] >= target - timedelta(days=tolerance_days)) &
        (orbit_catalogue['date'] <= target + timedelta(days=tolerance_days))
    ]
    if len(window) == 0:
        return None
    idx = (window['date'] - target).abs().idxmin()
    return window.loc[idx]


def check_valid_baseline(date1, date2):
    """Check that the temporal baseline between two SLC acquisitions is 6 or 12 days."""
    if date1 is None or date2 is None:
        return False
    delta = abs((pd.Timestamp(date2) - pd.Timestamp(date1)).days)
    return delta in VALID_BASELINES_DAYS


# Build the SLC triplet index
slc_index_records = []

for _, row in matches_df.iterrows():
    match_id = int(row['match_id'])

    slc_t1 = find_nearest_slc(row['before_far_date'])
    slc_t2 = find_nearest_slc(row['before_near_date'])
    slc_t3 = find_nearest_slc(row['after_date'])

    # Fallback: if t2 and t3 are from the same date, find the next acquisition for t3
    if slc_t2 is not None and slc_t3 is not None:
        if slc_t2['date'] == slc_t3['date']:
            # Look for t3 further into the future
            further = orbit_catalogue[
                orbit_catalogue['date'] > slc_t2['date'] + timedelta(days=1)
            ]
            if len(further) > 0:
                slc_t3 = further.iloc[0]

    t1_date = slc_t1['date'] if slc_t1 is not None else pd.NaT
    t2_date = slc_t2['date'] if slc_t2 is not None else pd.NaT
    t3_date = slc_t3['date'] if slc_t3 is not None else pd.NaT

    t1_id  = slc_t1['product_id'] if slc_t1 is not None else None
    t2_id  = slc_t2['product_id'] if slc_t2 is not None else None
    t3_id  = slc_t3['product_id'] if slc_t3 is not None else None

    t1_name = slc_t1['name'] if slc_t1 is not None else None
    t2_name = slc_t2['name'] if slc_t2 is not None else None
    t3_name = slc_t3['name'] if slc_t3 is not None else None

    baseline_before = abs((t2_date - t1_date).days) if pd.notna(t1_date) and pd.notna(t2_date) else None
    baseline_after  = abs((t3_date - t2_date).days) if pd.notna(t2_date) and pd.notna(t3_date) else None

    valid_before = baseline_before in VALID_BASELINES_DAYS if baseline_before is not None else False
    valid_after  = baseline_after  in VALID_BASELINES_DAYS if baseline_after  is not None else False
    full_ok      = valid_before and valid_after

    slc_index_records.append({
        'match_id':         match_id,
        'event_date_str':   row['event_date_str'],
        # S2 triplet dates
        's2_before_far':    row['before_far_date'].date(),
        's2_before_near':   row['before_near_date'].date(),
        's2_after':         row['after_date'].date(),
        # SLC triplet dates (may differ from S2 dates)
        'slc_t1_date':      t1_date.date() if pd.notna(t1_date) else None,
        'slc_t2_date':      t2_date.date() if pd.notna(t2_date) else None,
        'slc_t3_date':      t3_date.date() if pd.notna(t3_date) else None,
        # CDSE product IDs
        'slc_t1_id':        t1_id,
        'slc_t2_id':        t2_id,
        'slc_t3_id':        t3_id,
        # Product file names
        'slc_t1_name':      t1_name,
        'slc_t2_name':      t2_name,
        'slc_t3_name':      t3_name,
        # Temporal baselines
        'baseline_before':  baseline_before,
        'baseline_after':   baseline_after,
        # Validity
        'valid_before_pair': valid_before,
        'valid_after_pair':  valid_after,
        'full_triplet_ok':   full_ok,
    })

slc_index_df = pd.DataFrame(slc_index_records)

n_full  = slc_index_df['full_triplet_ok'].sum()
n_total = len(slc_index_df)
print(f'=== SLC Triplet Matching Summary ===')
print(f'Events with full valid SLC triplet:   {n_full} / {n_total}  ({n_full/n_total*100:.0f}%)')
print(f'Events with valid before pair only:   {slc_index_df["valid_before_pair"].sum()}')
print(f'Events with valid after pair only:    {slc_index_df["valid_after_pair"].sum()}')

print('\nBaseline distribution (before pair):')
print(slc_index_df['baseline_before'].value_counts().sort_index())
print('\nBaseline distribution (after pair):')
print(slc_index_df['baseline_after'].value_counts().sort_index())

# Show events that failed matching
failed = slc_index_df[~slc_index_df['full_triplet_ok']]
if len(failed) > 0:
    print(f'\n=== {len(failed)} events WITHOUT a valid SLC triplet ===')
    cols = ['event_date_str', 'baseline_before', 'baseline_after', 'valid_before_pair', 'valid_after_pair']
    print(failed[cols].to_string(index=False))

=== SLC Triplet Matching Summary ===
Events with full valid SLC triplet:   77 / 92  (84%)
Events with valid before pair only:   77
Events with valid after pair only:    91

Baseline distribution (before pair):
baseline_before
0     15
6     46
12    31
Name: count, dtype: int64

Baseline distribution (after pair):
baseline_after
6     56
12    35
18     1
Name: count, dtype: int64

=== 15 events WITHOUT a valid SLC triplet ===
 event_date_str  baseline_before  baseline_after  valid_before_pair  valid_after_pair
       20190626                0              12              False              True
       20190627                0              12              False              True
       20190725                0               6              False              True
       20190824                0              12              False              True
       20190827                0              12              False              True
       20190911                0               6     

---
## 7. Build Scene Index and Coverage Table

Save the full triplet index and a simplified coverage table for use in notebook 08.

In [28]:
# Unique SLC scenes needed (de-duplicate — many events share t2)
all_ids   = pd.concat([
    slc_index_df[slc_index_df['full_triplet_ok']][['slc_t1_id', 'slc_t1_name', 'slc_t1_date']].rename(columns={'slc_t1_id': 'product_id', 'slc_t1_name': 'name', 'slc_t1_date': 'date'}),
    slc_index_df[slc_index_df['full_triplet_ok']][['slc_t2_id', 'slc_t2_name', 'slc_t2_date']].rename(columns={'slc_t2_id': 'product_id', 'slc_t2_name': 'name', 'slc_t2_date': 'date'}),
    slc_index_df[slc_index_df['full_triplet_ok']][['slc_t3_id', 'slc_t3_name', 'slc_t3_date']].rename(columns={'slc_t3_id': 'product_id', 'slc_t3_name': 'name', 'slc_t3_date': 'date'}),
]).dropna(subset=['product_id']).drop_duplicates(subset='product_id').sort_values('date').reset_index(drop=True)

print(f'Unique SLC scenes to download: {len(all_ids)}')
print(f'(From {n_full} events with valid triplets)')
print(f'Estimated disk space: {len(all_ids) * 4:.0f} GB (at ~4 GB per scene)')

# Save
slc_index_df.to_csv(SLC_INDEX_PATH, index=False)
print(f'\nSaved: {SLC_INDEX_PATH}')

# Simplified coverage table (matches the pattern of s1_event_coverage.csv from nb02)
coverage_df = slc_index_df[[
    'match_id', 'event_date_str',
    'slc_t1_date', 'slc_t2_date', 'slc_t3_date',
    'slc_t1_id',  'slc_t2_id',  'slc_t3_id',
    'baseline_before', 'baseline_after',
    'valid_before_pair', 'valid_after_pair', 'full_triplet_ok',
]].copy()
coverage_df.to_csv(SLC_COVERAGE_PATH, index=False)
print(f'Saved: {SLC_COVERAGE_PATH}')

Unique SLC scenes to download: 64
(From 77 events with valid triplets)
Estimated disk space: 256 GB (at ~4 GB per scene)

Saved: ..\data\slc_scene_index.csv
Saved: ..\data\slc_event_coverage.csv


---
## 8. Download SLC Scenes (Batch Mode)

**Disk space management:**
- `BATCH_SIZE` controls how many scenes to download per run.
- After each batch, run notebook 08 to process the downloaded scenes,
  then manually delete the raw `.SAFE` directories from `data/Sentinel_S1_SLC/`.
- Scenes already on disk are automatically skipped.

**Expected total:** see cell above. At ~4 GB/scene and 350 GB free,
process in batches of ≤80 scenes.

In [29]:
def slc_safe_name(product_name):
    """Convert product name to expected .SAFE directory name."""
    if not product_name:
        return None
    name = product_name.replace('.zip', '').replace('.SAFE', '')
    return name + '.SAFE'


def is_on_disk(product_name, slc_dir=SLC_DIR):
    """Check if an SLC scene is already downloaded (.SAFE dir, .zip, or .SAFE.zip)."""
    if not product_name:
        return False
    safe_name = slc_safe_name(product_name)          # e.g. S1A_...SAFE
    return (
        os.path.isdir( os.path.join(slc_dir, safe_name))           or  # extracted
        os.path.isfile(os.path.join(slc_dir, safe_name + '.zip'))   or  # CDSE download
        os.path.isfile(os.path.join(slc_dir, safe_name.replace('.SAFE', '.zip')))  # plain zip
    )


def is_coherence_done(match_id, coh_dir=COH_DIR):
    """Return True if both coherence TIFs for this event are already on disk."""
    before = os.path.join(coh_dir, f'{int(match_id)}_coh_before.tif')
    after  = os.path.join(coh_dir, f'{int(match_id)}_coh_after.tif')
    return os.path.isfile(before) and os.path.isfile(after)


# Scenes still needed: not on disk AND referenced by at least one unprocessed event
def _scene_still_needed(product_id):
    refs = slc_index_df[
        slc_index_df['full_triplet_ok'] & (
            (slc_index_df['slc_t1_id'] == product_id) |
            (slc_index_df['slc_t2_id'] == product_id) |
            (slc_index_df['slc_t3_id'] == product_id)
        )
    ]
    return any(not is_coherence_done(mid) for mid in refs['match_id'])


# ── Download configuration ─────────────────────────────────────────────────
PILOT_MODE = False    # <- set False once pilot passes
BATCH_SIZE = 64      # covers all remaining scenes in one run

# Determine which scenes are still needed
to_download_all = [
    row for _, row in all_ids.iterrows()
    if not is_on_disk(row['name']) and _scene_still_needed(row['product_id'])
]
already_done = len(all_ids) - len(to_download_all)

if PILOT_MODE:
    first_event = slc_index_df[slc_index_df['full_triplet_ok']].iloc[0]
    pilot_ids   = {first_event['slc_t1_id'], first_event['slc_t2_id'], first_event['slc_t3_id']}
    to_download = [r for r in to_download_all if r['product_id'] in pilot_ids]
    print(f'PILOT MODE: downloading {len(to_download)} scenes for event '
          f'{first_event["event_date_str"]}  (match_id={int(first_event["match_id"])})')
    print(f'  t1: {first_event["slc_t1_name"]}')
    print(f'  t2: {first_event["slc_t2_name"]}')
    print(f'  t3: {first_event["slc_t3_name"]}')
else:
    to_download = to_download_all[:BATCH_SIZE]

print(f'Total unique scenes:         {len(all_ids)}')
print(f'Already processed (skipped): {already_done}')
print(f'This run will download:      {len(to_download)}')
print(f'Estimated size:              ~{len(to_download) * 7}-{len(to_download) * 8} GB  (~7-8 GB/scene)')


Total unique scenes:         64
Already processed (skipped): 33
This run will download:      31
Estimated size:              ~217-248 GB  (~7-8 GB/scene)


In [30]:
# Download the current batch
batch = to_download[:BATCH_SIZE] if not PILOT_MODE else to_download

if not batch:
    print('All scenes already on disk. Nothing to download.')
else:
    print(f'Starting download of {len(batch)} scenes to {SLC_DIR}...')
    print('This will take several hours. Monitor disk space during download.')
    print()

    # Build list of feature dicts for download_features()
    batch_ids     = {row['product_id'] for row in batch}
    batch_features = [f for f in catalogue_raw if f['Id'] in batch_ids]

    if len(batch_features) != len(batch):
        print(f'WARNING: only {len(batch_features)} / {len(batch)} features found in catalogue cache.')

    # download_features(features, path, options) — credentials go in the options dict
    download_options = {'credentials': credentials}

    downloaded = 0
    errors     = []

    for feat in batch_features:
        name = feat.get('Name', feat.get('Id', 'unknown'))
        print(f'  [{downloaded+1}/{len(batch_features)}] {name} ...', end=' ', flush=True)
        try:
            list(download_features([feat], SLC_DIR, download_options))
            downloaded += 1
            print('OK')
        except Exception as e:
            errors.append({'name': name, 'error': str(e)})
            print(f'ERROR: {e}')

    print(f'\nDownloaded: {downloaded}  Errors: {len(errors)}')
    if errors:
        print('\nFailed downloads:')
        for e in errors:
            print(f'  {e["name"]}: {e["error"]}')

Starting download of 31 scenes to ..\data\Sentinel_S1_SLC...
This will take several hours. Monitor disk space during download.

  [1/31] S1A_IW_SLC__1SDV_20200718T171526_20200718T171552_033512_03E220_2612.SAFE ... OK
  [2/31] S1B_IW_SLC__1SDV_20200724T171456_20200724T171523_022616_02AEC5_C385.SAFE ... OK
  [3/31] S1A_IW_SLC__1SDV_20200730T171551_20200730T171618_033687_03E77E_0CA0.SAFE ... OK
  [4/31] S1B_IW_SLC__1SDV_20200805T171457_20200805T171524_022791_02B417_A272.SAFE ... OK
  [5/31] S1A_IW_SLC__1SDV_20200811T171527_20200811T171554_033862_03ED41_5AC1.SAFE ... OK
  [6/31] S1B_IW_SLC__1SDV_20200817T171500_20200817T171527_022966_02B980_8E58.SAFE ... OK
  [7/31] S1A_IW_SLC__1SDV_20200823T171553_20200823T171619_034037_03F36B_2248.SAFE ... OK
  [8/31] S1A_IW_SLC__1SDV_20200904T171528_20200904T171555_034212_03F991_92EB.SAFE ... OK
  [9/31] S1B_IW_SLC__1SDV_20200910T171459_20200910T171526_023316_02C47B_946A.SAFE ... OK
  [10/31] S1B_IW_SLC__1SDV_20200922T171459_20200922T171526_023491_02C9F

---
## 9. Verify Downloaded Scenes

In [31]:
# Re-check what is on disk after download
on_disk_after = [row['name'] for _, row in all_ids.iterrows() if is_on_disk(row['name'])]
on_disk_set   = set(on_disk_after)

coverage_df['t1_on_disk'] = coverage_df['slc_t1_id'].apply(
    lambda pid: any(r['name'] in on_disk_set for _, r in all_ids.iterrows() if r['product_id'] == pid)
)
coverage_df['t2_on_disk'] = coverage_df['slc_t2_id'].apply(
    lambda pid: any(r['name'] in on_disk_set for _, r in all_ids.iterrows() if r['product_id'] == pid)
)
coverage_df['t3_on_disk'] = coverage_df['slc_t3_id'].apply(
    lambda pid: any(r['name'] in on_disk_set for _, r in all_ids.iterrows() if r['product_id'] == pid)
)
coverage_df['fully_ready'] = (
    coverage_df['full_triplet_ok'] &
    coverage_df['t1_on_disk'] &
    coverage_df['t2_on_disk'] &
    coverage_df['t3_on_disk']
)

n_ready = coverage_df['fully_ready'].sum()
n_total = len(coverage_df)

print(f'Events fully ready for coherence processing: {n_ready} / {n_total}  ({n_ready/n_total*100:.0f}%)')
if n_ready < n_total:
    not_ready = coverage_df[~coverage_df['fully_ready']]
    print(f'\n{len(not_ready)} events not yet ready (download remaining batches):')
    cols = ['event_date_str', 'full_triplet_ok', 't1_on_disk', 't2_on_disk', 't3_on_disk']
    print(not_ready[cols].head(20).to_string(index=False))

# Update coverage file with disk status
coverage_df.to_csv(SLC_COVERAGE_PATH, index=False)
print(f'\nUpdated: {SLC_COVERAGE_PATH}')

Events fully ready for coherence processing: 32 / 92  (35%)

60 events not yet ready (download remaining batches):
 event_date_str  full_triplet_ok  t1_on_disk  t2_on_disk  t3_on_disk
       20190509             True       False       False       False
       20190516             True       False       False       False
       20190523             True       False       False       False
       20190530             True       False       False       False
       20190531             True       False       False       False
       20190605             True       False       False       False
       20190606             True       False       False       False
       20190616             True       False       False       False
       20190617             True       False       False       False
       20190618             True       False       False       False
       20190619             True       False       False       False
       20190620             True       False       False 

---
## Summary

**What this notebook does:**

1. Queries the CDSE catalogue for S1 IW SLC ascending products over the study AOI (2019–2023).
2. Identifies the dominant relative orbit track and groups acquisitions by 6/12-day pairs.
3. For each of the 92 mowing events, finds the best SLC triplet (t1, t2, t3) with valid
   temporal baselines for coherence_before and coherence_after.
4. Downloads SLC scenes in batches to stay within available disk space.

**Outputs:**
- `data/slc_scene_index.csv` — full triplet plan per event (SLC dates, product IDs, baselines)
- `data/slc_event_coverage.csv` — per-event readiness status (disk check included)
- `data/Sentinel_S1_SLC/*.SAFE` — downloaded SLC scenes
- `data/slc_catalogue_cache.json` — CDSE metadata cache (avoids repeated API queries)

**Next step → notebook 08:** Process each SLC pair through the SNAP coherence pipeline
(coregistration → coherence estimation → terrain correction → export at 20m EPSG:2056).

**Disk management reminder:**
After running notebook 08 on the first batch, delete the processed `.SAFE` directories
from `data/Sentinel_S1_SLC/`, then re-run cells 8–9 of this notebook to download the next batch.